In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
ROOT = Path.cwd().parents[1]
DATA = ROOT / "data" / "raw" / "movielens"


In [2]:
df = pd.read_csv(ROOT / "notebooks" / "02-preprocessing" / "movies_preprocessed_clean.csv", index_col=0)
tags = pd.read_csv(DATA / "tags.csv")
links = pd.read_csv(DATA / "links.csv")
cast = pd.read_csv(ROOT / "notebooks" / "02-preprocessing" /  "cast_group_clean.csv", index_col=0)

Avant tout on regarde si il manque de la donnée

In [3]:
tags.isna().sum()

userId        0
movieId       0
tag          16
timestamp     0
dtype: int64

In [4]:
tags = tags.dropna(subset=['tag'])

In [5]:
tags.head()

,userId,movieId,tag,timestamp
0,18,4141,Mark Waters,1240597180
1,65,208,dark hero,1368150078
2,65,353,dark hero,1368150079
3,65,521,noir thriller,1368149983
4,65,592,dark hero,1368150078


In [6]:
tags['movieId'].value_counts()

movieId
296      1994
2959     1779
79132    1552
2571     1430
318      1339
         ... 
8642        1
67894       1
73501       1
84553       1
1495        1
Name: count, Length: 19545, dtype: int64

In [7]:
tags['tag'].value_counts().head(30)

tag
sci-fi                3384
based on a book       3281
atmospheric           2917
comedy                2779
action                2657
surreal               2427
BD-R                  2334
twist ending          2323
funny                 2072
dystopia              1991
stylized              1941
quirky                1906
dark comedy           1899
classic               1769
psychology            1754
fantasy               1703
time travel           1549
romance               1534
visually appealing    1509
disturbing            1487
aliens                1428
thought-provoking     1422
social commentary     1417
Nudity (Topless)      1400
violence              1336
drugs                 1312
Criterion             1286
true story            1276
nudity (topless)      1245
adventure             1243
Name: count, dtype: int64

On voit clairement qu'il y as enormement de doublons de tag, qui sont liés uniquement a une majuscule presente dans un groupe et pas dans l'autre,
je vais donc tout standardiser, tout en minuscule + strip 

In [8]:
tags['tag'] = tags["tag"].str.lower().str.strip()

In [9]:
tags['tag'].value_counts().head(30)

tag
sci-fi                3576
based on a book       3307
atmospheric           3169
comedy                3078
action                3068
nudity (topless)      2646
surreal               2528
twist ending          2367
bd-r                  2334
funny                 2253
quirky                2034
dystopia              2015
classic               1971
stylized              1944
dark comedy           1910
romance               1874
fantasy               1850
psychology            1763
time travel           1572
disturbing            1524
visually appealing    1511
aliens                1439
social commentary     1424
thought-provoking     1422
adventure             1374
violence              1361
criterion             1352
animation             1345
drugs                 1336
dark                  1312
Name: count, dtype: int64

Le set est beaucoup mieux organisé, je vais regrouper tout ca pour une ligne de mot par films

In [10]:
tags_agg = tags.groupby('movieId')['tag'].agg(' '.join).reset_index()

In [11]:
tags_agg.head()

,movieId,tag
0,1,watched computer animation disney animated fea...
1,2,time travel adapted from:book board game child...
2,3,old people that is actually funny sequel fever...
3,4,chick flick revenge characters chick flick cha...
4,5,diane keaton family sequel steve martin weddin...


Le fichier links va faire le lien entre les 2 DF 

In [12]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [13]:
tags_links = tags_agg.merge(links, on="movieId", how="inner")

tags_links.head()
tags_links.shape

(19545, 4)

In [14]:
tags_links["imdbId"] = "tt" + tags_links['imdbId'].astype(str).str.zfill(7)



In [15]:
tags_links['imdbId'].head()


0    tt0114709
1    tt0113497
2    tt0113228
3    tt0114885
4    tt0113041
Name: imdbId, dtype: str

In [16]:
df = df.reset_index()


Je decide de garder tout les films memes sans tag movielens, le but etant d'enrichire les films avec certain tag utilisateur

In [17]:
final = df.merge(tags_links, left_on='tconst', right_on='imdbId', how='left')

In [18]:
final.head()

,tconst,primaryTitle,startYear,runtimeMinutes,averageRating,numVotes,weight_rating,Action,Adult,Adventure,...,Sci-Fi,Sport,Talk-Show,Thriller,War,Western,movieId,tag,imdbId,tmdbId
0,tt0000009,Miss Jerry,1894.0,45.0,5.3,236.0,6.125416,0,0,0,...,0,0,0,0,0,0,NaN,NaN,NaN,NaN
1,tt0000147,The Corbett-Fitzsimmons Fight,1897.0,100.0,5.3,600.0,6.081154,0,0,0,...,0,1,0,0,0,0,NaN,NaN,NaN,NaN
2,tt0000574,The Story of the Kelly Gang,1906.0,70.0,6.0,1067.0,6.133821,1,0,1,...,0,0,0,0,0,0,NaN,NaN,NaN,NaN
3,tt0000591,The Prodigal Son,1907.0,90.0,5.0,39.0,6.149650,0,0,0,...,0,0,0,0,0,0,NaN,NaN,NaN,NaN
4,tt0000615,Robbery Under Arms,1907.0,NaN,3.4,34.0,6.141831,0,0,0,...,0,0,0,0,0,0,NaN,NaN,NaN,NaN


In [19]:
final.shape

(330702, 38)

In [20]:
 
df.shape

(330702, 34)

In [21]:
final.columns

Index(['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes',
       'averageRating', 'numVotes', 'weight_rating', 'Action', 'Adult',
       'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary',
       'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History',
       'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV',
       'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western',
       'movieId', 'tag', 'imdbId', 'tmdbId'],
      dtype='str')

In [22]:
final['tag'].isna().sum()

np.int64(312824)

In [23]:
final['tag'] = final['tag'].fillna('')


Je merge mon cast avec mon df final

In [24]:
cast_final = final.merge(cast, how='left', on="tconst")

In [25]:
cast_final = cast_final.drop(["movieId", "imdbId", "tmdbId"], axis=1)

In [26]:
cast_final['tag'] = cast_final['tag'].fillna('')
cast_final['clean_name'] = cast_final['clean_name'].fillna('')

In [27]:
cast_final.shape

(330702, 36)

In [28]:

(cast_final['tag'] != '').sum()

np.int64(17878)

In [29]:
cast_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 330702 entries, 0 to 330701
Data columns (total 36 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          330702 non-null  str    
 1   primaryTitle    330700 non-null  str    
 2   startYear       330665 non-null  float64
 3   runtimeMinutes  301550 non-null  float64
 4   averageRating   330702 non-null  float64
 5   numVotes        330702 non-null  float64
 6   weight_rating   330702 non-null  float64
 7   Action          330702 non-null  int64  
 8   Adult           330702 non-null  int64  
 9   Adventure       330702 non-null  int64  
 10  Animation       330702 non-null  int64  
 11  Biography       330702 non-null  int64  
 12  Comedy          330702 non-null  int64  
 13  Crime           330702 non-null  int64  
 14  Documentary     330702 non-null  int64  
 15  Drama           330702 non-null  int64  
 16  Family          330702 non-null  int64  
 17  Fantasy         33070

# 17878 films enrichis en casting + tags utilisateurs

 Je crée 3 matrice, Genre, Cast et Tags que je normalise pour egalisé le poid

In [30]:
genre_cols = cast_final.loc[:, 'Action':'Western'].columns.tolist()

CV = CountVectorizer()

cast_sparse = CV.fit_transform(cast_final['clean_name'])
genre_sparse = csr_matrix(cast_final[genre_cols].values)

genre_matrice = normalize(genre_sparse)

cast_matrice = normalize(cast_sparse)

TFIDF = TfidfVectorizer()

tags_sparse = TFIDF.fit_transform(cast_final['tag'])

tags_matrice = normalize(tags_sparse)

In [31]:
genre_w = genre_matrice * 1.0
cast_w = cast_matrice * 1
tags_w = tags_matrice * 1

In [32]:
matrice = hstack([genre_w, cast_w, tags_w]).tocsr()

position = 0
scores = cosine_similarity(matrice[position], matrice)

scores = scores.flatten()
ordre = scores.argsort()[::-1]
top = ordre[1:11]

In [105]:
def recherche_par_titre(titre, n=10):
    matches = cast_final[cast_final['primaryTitle'].str.contains(titre, case=False, na=False)]
    if matches.empty:
        return f"Aucun film trouvé pour '{titre}'"
    position = matches.sort_values('weight_rating', ascending=False).index[0]

    sims = cosine_similarity(matrice[position], matrice).flatten()
    df_temp = cast_final.copy()
    df_temp['sim'] = sims

    df_temp = df_temp.drop(position)
    print(cast_final.loc[position, 'primaryTitle'])
    df_temp = df_temp.sort_values(['sim', 'weight_rating'], ascending=[False, False])
    return df_temp[['primaryTitle', 'startYear', 'clean_name', 'weight_rating', 'sim']].head(n)



In [106]:
cast_final.index.equals(pd.RangeIndex(len(cast_final)))

## index de cast_final doit rester 0..n sans trou (matrice[position] = position, pas étiquette)

True

In [107]:
recherche_par_titre('Matrix')

IndexError: index (69308) out of range

Les films sans cast sont avantager , on peut voir que la similarité est identique et le WR vratiment proche sur le 2 films sans cast 
RiffTrax: Firehead	2013.0		6.168302	0.808452 \n
    	Armageddon	1969.0		6.160124	0.808452

In [36]:
df['numVotes'].describe()

count    3.307020e+05
mean     3.825301e+03
std      3.838376e+04
min      5.000000e+00
25%      2.200000e+01
50%      7.000000e+01
75%      3.480000e+02
max      3.175959e+06
Name: numVotes, dtype: float64

In [37]:
df['numVotes'].quantile([0.8, 0.9, 0.92, 0.95, 0.99])

0.80      546.00
0.90     1943.00
0.92     2825.00
0.95     6187.95
0.99    77239.73
Name: numVotes, dtype: float64

On va filtrer par nombres de votes, sur le quantile 92 qui est a 2825 notes

In [38]:
cast_final_Q92 = cast_final[cast_final['numVotes'] >= df['numVotes'].quantile(0.92)].reset_index(drop=True)

In [39]:
cast_final_Q92.shape

(26463, 36)

In [119]:
genre_cols = cast_final_Q92.loc[:, 'Action':'Western'].columns.tolist()

CV = CountVectorizer()

cast_sparse = CV.fit_transform(cast_final_Q92['clean_name'])
genre_sparse = csr_matrix(cast_final_Q92[genre_cols].values)

genre_matrice = normalize(genre_sparse)

cast_matrice = normalize(cast_sparse)

TFIDF = TfidfVectorizer()

tags_sparse = TFIDF.fit_transform(cast_final_Q92['tag'])

tags_matrice = normalize(tags_sparse)

genre_w = genre_matrice * 1.0
cast_w = cast_matrice * 1
tags_w = tags_matrice * 1

matrice = hstack([genre_w, cast_w, tags_w]).tocsr()

def recherche_par_titre(titre, n=10):
    matches = cast_final_Q92[cast_final_Q92['primaryTitle'].str.contains(titre, case=False, na=False)]
    if matches.empty:
        return f"Aucun film trouvé pour '{titre}'"
    position = matches.sort_values('weight_rating', ascending=False).index[0]

    sims = cosine_similarity(matrice[position], matrice).flatten()
    df_temp = cast_final_Q92.copy()
    df_temp['sim'] = sims

    df_temp = df_temp.drop(position)
    
    df_temp = df_temp.sort_values(['sim'], ascending=[False])
    return df_temp[['primaryTitle', 'startYear', 'clean_name', 'weight_rating', 'sim' ]].head(n)


In [138]:
recherche_par_titre('Spirited ')

,primaryTitle,startYear,clean_name,weight_rating,sim
10326,Howl's Moving Castle,2004.0,chiekobaishô takuyakimura tatsuyagashûin akihi...,8.175498,0.726572
5052,Castle in the Sky,1986.0,mayumitanaka keikoyokozawa kotoehatsui minorit...,7.943994,0.634218
7734,Princess Mononoke,1997.0,yôjimatsuda yurikoishida yûkotanaka kaorukobay...,8.272830,0.538949
12795,Ponyo,2008.0,tomokoyamaguchi kazushigenagashima yûkiamami g...,7.553161,0.538584
3863,Lupin III: The Castle of Cagliostro,1979.0,yasuoyamada eikomasuyama kiyoshikobayashi maki...,7.387531,0.516943
6299,Porco Rosso,1992.0,shûichirômoriyama tokikokatô bunshikatsuravi t...,7.621796,0.515532
5640,Kiki's Delivery Service,1989.0,minamitakayama minamitakayama reisakuma miekon...,7.747447,0.507726
5503,My Neighbor Totoro,1988.0,hitoshitakagi norikohidaka chikasakamoto shige...,8.072395,0.505989
4594,Nausicaä of the Valley of the Wind,1984.0,sumishimamoto mahitotsujimura mahitotsujimura ...,7.943752,0.497109
16919,The Secret World of Arrietty,2010.0,miraishida ryûnosukekamiki tatsuyafujiwara tom...,7.527243,0.445300


Je vais faire une fonction de recherche par genre aussi     

In [130]:
def recherche_par_genre(genre, n=10):
    mapping = {g.lower(): g for g in genre_cols}
    genre_col = mapping.get(genre.lower())
    if genre_col is None:
          return f"Genre '{genre}' inconnu"

    matches = cast_final_Q92[genre_col] == 1
    films = cast_final_Q92[matches]
    if films.empty:
         return f"Aucun film trouvé pour '{genre}'"

    
    films_tries = films.sort_values('weight_rating', ascending=False)
    return films_tries[['primaryTitle', 'startYear', 'clean_name', 'weight_rating']].head(n)


In [133]:
print(genre_cols)

['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']


In [142]:
recherche_par_genre('Thriller')

,primaryTitle,startYear,clean_name,weight_rating
8176,Fight Club,1999.0,bradpitt edwardnorton meatloaf zachgrenier ric...,8.793710
11083,The Departed,2006.0,leonardodicaprio mattdamon jacknicholson markw...,8.490677
24758,Parasite,2019.0,songkang-ho leesun-kyun choyeo-jeong choiwoo-s...,8.487526
1252,Rear Window,1954.0,jamesstewart gracekelly wendellcorey thelmarit...,8.474742
23737,Mirror Game,2016.0,chanchalchowdhury masumarahmannabila parthabar...,8.442967
8822,Memento,2000.0,guypearce carrie-annemoss joepantoliano markbo...,8.390342
24253,Kill Bill: The Whole Bloody Affair,2004.0,lucyliu vivicaafox michaelmadsen darylhannah d...,8.387807
11049,The Lives of Others,2006.0,ulrichmühe martinagedeck sebastiankoch ulricht...,8.369050
25103,Joker,2019.0,joaquinphoenix robertdeniro zaziebeetz frances...,8.292163
6363,Reservoir Dogs,1992.0,harveykeitel harveykeitel timroth timroth mich...,8.288704
